# exp00r0 — baseline + noise measurement

**Round** 0 (setup; not an experiment round) · **Parent** none — this *is* the baseline

Round 0 produces the three things every later round depends on, and nothing else:

1. A **clean baseline run** on the upstream config. No changes, no tuning.
2. The platform's **throughput**, which sets `SHARD_BUDGET` so that no round
   trains for more than one epoch (`research-program.md` §1.2).
3. **σ₀** — the seed-to-seed standard deviation of `val_bpb`. Every later keep/discard
   verdict is measured against this. Without it there is no decision rule, only noise.

Do not tune anything here. Round 0 is instrumentation.

## 1 · Setup

`prepare_round()` must run **before** any dataloader or eval call — it is what repoints
`prepare.DATA_DIR` at the round's shard slice. The harness asserts if you forget.

In [ ]:
import sys, os
sys.path.insert(0, "/home/dangnguyenducanh/autoresearch")  # repo root, so `prepare` imports

import prepare

# --- §2 project constants. Pin ONCE for the whole project, never per experiment. ---
# Upstream defaults. Override here only if the platform forces it, and then never again.
# prepare.MAX_SEQ_LEN = 2048
# prepare.EVAL_TOKENS = 40 * 524288

from experiments._shared.data import prepare_round, required_shard_budget, epochs_per_round
from experiments._shared.harness import ExpConfig, run, run_seeds

prepare_round(0)
print(f"MAX_SEQ_LEN={prepare.MAX_SEQ_LEN}  EVAL_TOKENS={prepare.EVAL_TOKENS:,}")

## 2 · Baseline config

Upstream `train.py` values, verbatim. `gpu=` only affects the reported MFU figure, never
`val_bpb` — set it to your card so the MFU number means something.

In [ ]:
baseline = ExpConfig(
    round_idx=0,
    seed=0,
    gpu="H100",            # key into harness.PEAK_FLOPS; MFU reporting only
    notes="upstream train.py defaults, unmodified",
)
print(f"model_dim = {baseline.model_dim()}  grad_accum = {baseline.grad_accum_steps()}")
for k, v in vars(baseline).items():
    print(f"  {k:20s} {v}")

## 3 · One run — does it work, and how fast is this machine?

In [ ]:
m = run(baseline, outdir="experiments/exp00r0")
throughput = m["tok_per_s"]
print(f"\nthroughput: {throughput:,} tokens/s")

## 4 · Size the dataset — the epoch gate

Every round must stay under 1.0 epoch. A round that repeats data measures memorization
rather than the change under test, so its verdicts do not transfer to the next round.

If any round comes out ≥ 1.0, **raise `SHARD_BUDGET`** in `experiments/_shared/data.py`
and download more shards. Every round trains for the same `prepare.TIME_BUDGET`, so the
binding case is **R1** — the least data at the same wall clock. Size for R1 and the
larger rounds clear the gate automatically.

In [ ]:
needed = required_shard_budget(throughput)
print(f"SHARD_BUDGET needed for this platform: {needed}")
print(f"  -> set SHARD_BUDGET = {needed} in experiments/_shared/data.py")
print(f"  -> download with: uv run prepare.py --num-shards {needed}")
print(f"  -> TEST_SHARD 6541 must be fetched separately (prepare.py only pins 6542)\n")

eps = epochs_per_round(throughput, shard_budget=needed)
for r, e in sorted(eps.items()):
    print(f"  round {r}: {e:.3f} epochs  {'OK' if e < 1.0 else 'FAIL — raise SHARD_BUDGET'}")
assert all(e < 1.0 for e in eps.values()), "a round would repeat data; raise SHARD_BUDGET"

## 5 · σ₀ — the noise band

Three runs, identical config, seeds 0/1/2. The spread is the resolution limit of this
whole project: any `val_bpb` difference smaller than ~2σ is not a result.

Record σ₀ in `rounds/r1-directions.md`. Round 1 cannot open without it.

In [ ]:
mean0, sigma0, runs0 = run_seeds(baseline, seeds=(0, 1, 2), outdir="experiments/exp00r0")
print(f"\nBASELINE  mean val_bpb = {mean0:.6f}")
print(f"SIGMA_0   = {sigma0:.6f}")
print(f"keep threshold for R1 (-2σ): val_bpb < {mean0 - 2 * sigma0:.6f}")
print(f"\nIf sigma0 is 0.000000 the seeds are not actually varying — check that")
print(f"ExpConfig.seed reaches torch.manual_seed before fixing sigma for the round.")

## 6 · R0 exit gate

All three must hold before Round 1 opens (`research-program.md` §1.3):

- [ ] A run completes and prints `val_bpb:` without crashing
- [ ] `SHARD_BUDGET` computed, and every round under 1.0 epoch
- [ ] σ₀ measured from 3 seed repeats

Then: write `experiments/exp00r0/report.md`, agree `rounds/r1-directions.md` with the
human, and **stop**. Round 1 does not start without approval.